# ✅ 08 — Task Completion and Conversations

A support agent exists to get things done: a refund issued, a ticket opened, a question
answered. This notebook checks whether each task was actually completed, using the mock store
as ground truth, then asks two LLM judges the same question and measures how often they agree
with that ground truth. Finally it moves from single messages to whole conversations.

## Learning Objectives
In this notebook, you will learn:
1. **End-state checks** - comparing the store after a run with the state the case expects
2. **Task-completion judges** - DeepEval `TaskCompletionMetric` and RAGAS goal accuracy, against the state check
3. **Multi-turn evaluation** - scripted conversations, and a simulated customer that drives new ones
4. **Conversational metrics** - knowledge retention, completeness, turn relevancy and role adherence

## Prerequisites
- Notebooks 01 and 04 run once: they build `data/chroma_db/` and record `data/golden_runs.json`
- `OPENAI_API_KEY` and a LangSmith API key in the `.env` file at the repository root

---
## 🗺️ The idea

```
 single messages (golden runs)                     conversations (this notebook runs them)
 ─────────────────────────────                     ───────────────────────────────────────
 store after the run ──► end-state check ◄── truth  scripted turns ─┐
 trace + reply ──► TaskCompletion judge ─┐          simulated user ─┴─► agent ─► 4 conversational metrics
 messages + goal ──► RAGAS goal judge ───┴─► agree with the truth?
```

The lesson of the first half: when a task leaves a mark in the world (a refund row, a ticket),
**check the mark**. A judge reading the transcript can be talked into believing a refund
happened because the reply says so; the store can't.

---
## ⚙️ Part 0: Setup

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: keys, models, the golden set and the recorded runs
# ============================================================================
import os
import json
import warnings
from collections import Counter, defaultdict

from dotenv import load_dotenv

warnings.filterwarnings("ignore", message="IProgress not found")   # tqdm's note about Jupyter widgets
load_dotenv(os.path.join("..", "..", ".env"))
os.environ["DEEPEVAL_TELEMETRY_OPT_OUT"] = "YES"
os.environ["RAGAS_DO_NOT_TRACK"] = "true"
JUDGE_MODEL = "gpt-4o"
SIMULATED_USER_MODEL = "gpt-4o-mini"       # plays the customer in Part 6

RUNS_PATH = os.path.join("data", "golden_runs.json")
if not os.path.exists(RUNS_PATH):
    raise RuntimeError("No data/golden_runs.json. Run notebook 04 first: it records data/golden_runs.json.")

with open(os.path.join("data", "golden_set.json"), encoding="utf-8") as f:
    golden_set = json.load(f)
with open(RUNS_PATH, encoding="utf-8") as f:
    golden_runs = json.load(f)
runs_by_case = {run["case_id"]: run for run in golden_runs}

# The three agent slices are the ones whose cases say what the store must look like afterwards.
state_cases = [case for case in golden_set if case["expected_end_state"] is not None]
print("✅ Setup complete")
print(f"📄 {len(golden_set)} cases; {len(state_cases)} have an expected end state")

In [ ]:
# ============================================================================
# A SMALL REPORTING HELPER: pass rate per slice
# ============================================================================
SLICES = ["simple", "multi_hop", "out_of_scope", "tool_required", "handoff", "adversarial"]

def print_pass_rate_by_slice(results):
    """results: a list of (slice_name, passed) pairs. Prints one line per slice, then the total."""
    passed_by_slice, total_by_slice = Counter(), Counter()
    for slice_name, passed in results:
        total_by_slice[slice_name] += 1
        passed_by_slice[slice_name] += int(passed)
    for slice_name in SLICES:
        if total_by_slice[slice_name]:
            rate = passed_by_slice[slice_name] / total_by_slice[slice_name]
            print(f"   {slice_name:<14} {passed_by_slice[slice_name]:>2}/{total_by_slice[slice_name]:<3} {rate:.0%}")
    overall = sum(passed_by_slice.values()) / sum(total_by_slice.values())
    print(f"   {'all':<14} {sum(passed_by_slice.values()):>2}/{sum(total_by_slice.values()):<3} {overall:.0%}")

scores = defaultdict(dict)   # case_id → every score this notebook computes for it
print("✅ Helper ready")

---
## 🏁 Part 1: The end-state check

Each agent case says what the store must look like after the run: which orders were refunded
(`refund_order_ids`) and whether a ticket exists (`ticket_created`, `null` = not checked). The
run records what actually happened. Comparing them is free, exact and repeatable, and it is
the ground truth for the rest of this notebook.

In [ ]:
# ============================================================================
# END-STATE CHECK: refunds and tickets after the run, against the case
# ============================================================================
state_results = []
for case in state_cases:
    run, expected = runs_by_case[case["id"]], case["expected_end_state"]
    actual_refunds = sorted(refund["order_id"] for refund in run["refunds_issued"])
    refunds_ok = actual_refunds == sorted(expected["refund_order_ids"])
    ticket_ok = True
    if expected["ticket_created"] is not None:
        ticket_ok = (len(run["tickets_created"]) > 0) == expected["ticket_created"]
    passed = refunds_ok and ticket_ok
    scores[case["id"]]["end_state_correct"] = passed
    state_results.append((case["slice"], passed))
    if not passed:
        print(f"   ❌ {case['id']}: refunds {actual_refunds} (expected {expected['refund_order_ids']}), "
              f"tickets {len(run['tickets_created'])} (expected {expected['ticket_created']})")

print("🏁 End state matches the case")
print_pass_rate_by_slice(state_results)
print("Reading: this is the number to trust for 'did it do the job'. Every judge below is measured against it.")

---
## 📦 Part 2: The app, and the tool results

The judges need to see what each tool returned, and Part 5 runs the agent in conversations, so
the app is pasted in as notebook 02 Part 9 gives it. Tool results are rebuilt by replaying each
run's recorded calls against a fresh store, exactly as in notebook 06. Tracing is paused for the
replay, because those calls are not agent runs.

In [ ]:
# ============================================================================
# THE APP UNDER TEST: notebook 02's support agent, in one cell
# ============================================================================
# Copied from notebook 02, Part 9. Needs data/chroma_db/ (built by notebook 01).

# --- ENVIRONMENT SETUP: keys, tracing, models, paths
import os
import json
import time
import uuid
from datetime import date
from typing import Annotated, Literal, TypedDict

from dotenv import load_dotenv
from pydantic import BaseModel
import chromadb
from chromadb.errors import NotFoundError
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition

load_dotenv(os.path.join("..", "..", ".env"))
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "end-to-end-eval-tutorial"

APP_MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"
CHROMA_DIR = os.path.join("data", "chroma_db")

# --- OPEN THE SEARCH INDEX built by notebook 01
embedding_function = OpenAIEmbeddingFunction(model_name=EMBEDDING_MODEL, api_key_env_var="OPENAI_API_KEY")
chroma_client = chromadb.PersistentClient(path=CHROMA_DIR)

try:
    collection = chroma_client.get_collection("support_kb", embedding_function=embedding_function)
except NotFoundError:
    raise RuntimeError("No search index in data/chroma_db/. Run notebook 01 first: it builds it.")

# --- RETRIEVE AND ANSWER (same as notebook 01)
TOP_K = 4
llm = ChatOpenAI(model=APP_MODEL, temperature=0)

RAG_PROMPT = """You are the support assistant for Pinecrest Outdoor Supply.
Answer the customer's question using ONLY the support articles below.
After each fact, cite the chunk it came from in square brackets, for example [returns-policy-02].
If the articles do not contain the answer, reply exactly:
"I'm sorry, I don't have information about that. I can connect you with a human agent."

Support articles:
{context}"""

def retrieve(question, k=TOP_K):
    results = collection.query(query_texts=[question], n_results=k)
    hits = []
    for chunk_id, text in zip(results["ids"][0], results["documents"][0]):
        hits.append({"chunk_id": chunk_id, "text": text})
    return hits

def generate_answer(question, chunk_ids, contexts):
    labelled_chunks = []
    for chunk_id, text in zip(chunk_ids, contexts):
        labelled_chunks.append(f"[{chunk_id}]\n{text}")
    messages = [
        SystemMessage(content=RAG_PROMPT.format(context="\n\n".join(labelled_chunks))),
        HumanMessage(content=question),
    ]
    return llm.invoke(messages).content

# --- MOCK ORDER DATABASE: the "real world" the tools act on
TODAY = date(2026, 10, 18)

def reset_database():
    """A fresh copy of the store. Called before every agent run, so a refund in one
    test can never leak into the next one."""
    return {
        "orders": {
            # Delivered 10 days ago, full price: a normal, allowed refund.
            "PC-10001": {"email": "maya@example.com", "status": "delivered", "delivered_on": "2026-10-08",
                         "items": [{"name": "Ridge 2 Tent", "price": 249.00, "on_sale": False}]},
            # Sale item delivered 20 days ago: past the 14-day sale window, final sale.
            "PC-10002": {"email": "sam@example.com", "status": "delivered", "delivered_on": "2026-09-28",
                         "items": [{"name": "Storm Shell Jacket", "price": 129.00, "on_sale": True}]},
            # Not delivered yet: nothing to refund.
            "PC-10003": {"email": "lee@example.com", "status": "shipped", "delivered_on": None,
                         "items": [{"name": "Trail Pack 30L", "price": 89.00, "on_sale": False}]},
            # Above the $500 limit: the agent must hand over to a human.
            "PC-10004": {"email": "ana@example.com", "status": "delivered", "delivered_on": "2026-10-15",
                         "items": [{"name": "Summit Down Sleeping Bag", "price": 649.00, "on_sale": False}]},
            # Still processing: can be changed or cancelled.
            "PC-10005": {"email": "raj@example.com", "status": "processing", "delivered_on": None,
                         "items": [{"name": "Carbon Trekking Poles", "price": 79.00, "on_sale": False}]},
            # Delivered 47 days ago: outside the 30-day window, a human must decide.
            "PC-10006": {"email": "maya@example.com", "status": "delivered", "delivered_on": "2026-09-01",
                         "items": [{"name": "Compact Camp Stove", "price": 59.00, "on_sale": False}]},
        },
        "refunds": [],   # every refund the agent issues lands here
        "tickets": [],   # every handover to a human lands here
    }

database = reset_database()

# --- TOOL: lookup_order
@tool
def lookup_order(order_id: str, email: str) -> dict:
    """Look up an order's status, items and delivery date.
    Needs the order ID (like PC-10001) and the email address on the order."""
    order = database["orders"].get(order_id)
    if order is None:
        return {"error": f"No order found with ID {order_id}."}
    # Privacy rule from the account-and-privacy article: never reveal an order to
    # someone who can't confirm the email on it. Notebook 09 attacks this check.
    if order["email"].lower() != email.lower():
        return {"error": "The email does not match this order."}
    return {"order_id": order_id, "status": order["status"],
            "delivered_on": order["delivered_on"], "items": order["items"]}

# --- TOOL: issue_refund — the one tool that changes the world, so it checks the rules
@tool
def issue_refund(order_id: str, email: str, reason: str) -> dict:
    """Refund a delivered order. Needs the order ID, the email on the order and the
    customer's reason. Works only inside the return window and up to $500."""
    order = database["orders"].get(order_id)
    if order is None or order["email"].lower() != email.lower():
        return {"refunded": False, "why": "Order not found, or the email does not match."}
    if order["status"] != "delivered":
        return {"refunded": False, "why": f"The order is '{order['status']}', not delivered."}

    amount = sum(item["price"] for item in order["items"])
    days_since_delivery = (TODAY - date.fromisoformat(order["delivered_on"])).days
    on_sale = any(item["on_sale"] for item in order["items"])

    # The rules below are the same ones the support articles state.
    if on_sale and days_since_delivery > 14:
        return {"refunded": False, "why": f"Sale item delivered {days_since_delivery} days ago: final sale after 14 days."}
    if days_since_delivery > 30:
        return {"refunded": False, "needs_human": True,
                "why": f"Delivered {days_since_delivery} days ago, outside the 30-day window."}
    if amount > 500:
        return {"refunded": False, "needs_human": True, "why": f"${amount:.2f} is above the $500 limit."}

    database["refunds"].append({"order_id": order_id, "amount": amount, "reason": reason})
    order["status"] = "refunded"
    return {"refunded": True, "amount": amount}

# --- TOOLS: create_ticket and search_kb
@tool
def create_ticket(summary: str, priority: str = "normal") -> dict:
    """Hand the conversation to a human support agent. Give a one-line summary of what
    the customer needs. Priority is "normal" or "urgent" (injuries, safety, legal)."""
    ticket_id = f"T-{1001 + len(database['tickets'])}"
    database["tickets"].append({"ticket_id": ticket_id, "summary": summary, "priority": priority})
    return {"ticket_id": ticket_id}

@tool
def search_kb(query: str) -> list:
    """Search the support articles for a policy (returns, refunds, shipping, warranty...)."""
    hits = retrieve(query, k=3)
    return [{"chunk_id": hit["chunk_id"], "text": hit["text"]} for hit in hits]

ACCOUNT_TOOLS = [lookup_order, issue_refund, create_ticket, search_kb]

# Tools can be called directly too, which is handy for checking the rules by hand.
database = reset_database()   # undo anything the checks above changed

# --- ACCOUNT AGENT: the model, its instructions, and the node that runs the tools
account_llm = ChatOpenAI(model=APP_MODEL, temperature=0).bind_tools(ACCOUNT_TOOLS)

ACCOUNT_PROMPT = """You are the account assistant for Pinecrest Outdoor Supply. You handle orders and refunds.
Rules:
- To look up or refund an order you need BOTH the order ID and the email on the order.
  If either is missing, ask the customer for it and call no tool.
- If the customer already gave both, call the tool straight away. Never ask them to
  confirm details they have already given; the tool checks that the email matches.
- Never share details of an order whose email does not match.
- If issue_refund returns needs_human, call create_ticket with a one-line summary, then tell
  the customer a human agent will reply by email within 1 business day.
- Use search_kb to check a policy before explaining it.
Keep replies short and friendly."""

def account_agent_node(state):
    messages = [SystemMessage(content=ACCOUNT_PROMPT)] + state["messages"]
    response = account_llm.invoke(messages)
    # While the model still asks for tools, it has not answered the customer yet.
    answer = "" if response.tool_calls else response.content
    return {"messages": [response], "answer": answer}

account_tools_node = ToolNode(ACCOUNT_TOOLS)   # runs whichever tool the model asked for

# --- SUPERVISOR: pick the route; ESCALATION: open a ticket for a human
# LangChain turns this class into a JSON schema the model must fill in,
# so the answer is always one of the three routes, never free text.
class RouteDecision(BaseModel):
    route: Literal["rag_agent", "account_agent", "escalation"]
    reason: str

SUPERVISOR_PROMPT = """You route customer messages for Pinecrest Outdoor Supply's support desk.
- rag_agent: general questions about how things work: return windows, order statuses,
  shipping, warranty terms, loyalty points or product care. Use it when the customer gives
  no order ID, even if they mention something they bought.
- account_agent: the customer gives an order ID (like PC-10001) or asks to check, change,
  refund or return one specific order of theirs.
- escalation: the customer asks for a human, reports an injury or safety incident,
  makes a legal complaint, or wants to make a warranty claim."""

router_llm = ChatOpenAI(model=APP_MODEL, temperature=0).with_structured_output(RouteDecision)

def supervisor_node(state):
    decision = router_llm.invoke([SystemMessage(content=SUPERVISOR_PROMPT), HumanMessage(content=state["question"])])
    return {"route": decision.route}

def escalation_node(state):
    ticket = create_ticket.invoke({"summary": state["question"][:200], "priority": "urgent"})
    answer = (f"I've passed this to a human support agent (ticket {ticket['ticket_id']}). "
              "They will reply by email within 1 business day.")
    return {"answer": answer}

# --- THE SUPPORT AGENT GRAPH
class SupportState(TypedDict):
    question: str
    messages: Annotated[list, add_messages]   # the account agent's conversation with its tools
    route: str                                # the supervisor's decision → routing accuracy
    retrieved_chunk_ids: list                 # RAG path only → retrieval metrics
    contexts: list                            # RAG path only → faithfulness
    answer: str                               # the final reply to the customer

def retrieve_node(state):
    hits = retrieve(state["question"])
    return {"retrieved_chunk_ids": [hit["chunk_id"] for hit in hits],
            "contexts": [hit["text"] for hit in hits]}

def generate_node(state):
    return {"answer": generate_answer(state["question"], state["retrieved_chunk_ids"], state["contexts"])}

def route_after_supervisor(state):
    return state["route"]   # the name of the next node

builder = StateGraph(SupportState)
builder.add_node("supervisor", supervisor_node)
builder.add_node("retrieve", retrieve_node)
builder.add_node("generate", generate_node)
builder.add_node("account_agent", account_agent_node)
builder.add_node("account_tools", account_tools_node)
builder.add_node("escalation", escalation_node)

builder.add_edge(START, "supervisor")
builder.add_conditional_edges("supervisor", route_after_supervisor,
                              {"rag_agent": "retrieve", "account_agent": "account_agent", "escalation": "escalation"})
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", END)
# The loop: if the model asked for a tool, run it and come back; otherwise finish.
builder.add_conditional_edges("account_agent", tools_condition, {"tools": "account_tools", END: END})
builder.add_edge("account_tools", "account_agent")
builder.add_edge("escalation", END)

support_agent = builder.compile()

# --- RUN ONE QUESTION AND RECORD WHAT HAPPENED INSIDE
def run_agent(question):
    global database
    database = reset_database()   # every run starts from the same store

    run_id = uuid.uuid4()         # the same ID is on the LangSmith trace
    nodes_visited, tool_calls, final_state = [], [], {}
    start = time.time()

    # stream_mode="updates" gives one update per node, in the order the nodes ran.
    for update in support_agent.stream(
        {"question": question, "messages": [HumanMessage(content=question)]},
        config={"run_id": run_id, "run_name": "support-agent", "tags": ["notebook-02"]},
        stream_mode="updates",
    ):
        for node_name, node_output in update.items():
            nodes_visited.append(node_name)
            node_output = node_output or {}
            for message in node_output.get("messages", []):
                for call in getattr(message, "tool_calls", []):
                    tool_calls.append({"name": call["name"], "args": call["args"]})
            final_state.update(node_output)

    return {
        "run_id": str(run_id),
        "question": question,
        "route": final_state.get("route"),
        "nodes_visited": nodes_visited,
        "tool_calls": tool_calls,
        "retrieved_chunk_ids": final_state.get("retrieved_chunk_ids", []),
        "contexts": final_state.get("contexts", []),
        "answer": final_state.get("answer", ""),
        "refunds_issued": database["refunds"],    # the end state → task completion, notebook 08
        "tickets_created": database["tickets"],
        "latency_seconds": round(time.time() - start, 2),
    }

print("✅ App ready: support_agent, run_agent(question), database")

In [ ]:
# ============================================================================
# REPLAY: rebuild every tool result from the recorded calls (tracing paused)
# ============================================================================
os.environ["LANGSMITH_TRACING"] = "false"          # a replay is not an agent run
tools_by_name = {tool_object.name: tool_object for tool_object in ACCOUNT_TOOLS}

def replay_tool_calls(run):
    """Reset the store, re-run the run's tool calls in order, return their results."""
    global database
    database = reset_database()
    results = []
    for call in run["tool_calls"]:
        results.append(tools_by_name[call["name"]].invoke(call["args"]))
    return results

replayed_results = {}
for case in state_cases:
    replayed_results[case["id"]] = replay_tool_calls(runs_by_case[case["id"]])
os.environ["LANGSMITH_TRACING"] = "true"           # back on for the conversations in Part 5
print(f"✅ Replayed the tool calls of {len(replayed_results)} runs")

---
## ⚖️ Part 3: DeepEval `TaskCompletionMetric`

The judge reads the customer's message, the tool calls with their results, and the reply. It
works out what the task was and how much of it was done, and returns a score from 0 to 1. It
runs on all 60 agent cases, so every disagreement with the state check is visible.

In [ ]:
# ============================================================================
# DEEPEVAL TaskCompletionMetric on every agent case
# ============================================================================
# 💰 60 cases × 2 judge calls with gpt-4o: about $0.40.
from deepeval.test_case import LLMTestCase, ToolCall
from deepeval.metrics import TaskCompletionMetric

task_completion = TaskCompletionMetric(model=JUDGE_MODEL, async_mode=False)
for case in state_cases:
    run = runs_by_case[case["id"]]
    test_case = LLMTestCase(
        input=case["input"],
        actual_output=run["answer"],
        tools_called=[ToolCall(name=call["name"], input_parameters=call["args"], output=result)
                      for call, result in zip(run["tool_calls"], replayed_results[case["id"]])],
    )
    task_completion.measure(test_case, _show_indicator=False)   # no progress bar per case
    scores[case["id"]]["judge_task_completion"] = task_completion.score

judge_results = [(case["slice"], scores[case["id"]]["judge_task_completion"] >= 0.5) for case in state_cases]
print("⚖️ TaskCompletion judge says the task was done (score ≥ 0.5)")
print_pass_rate_by_slice(judge_results)

### 3.1 🎯 Telling the judge what "done" means

Left alone, the judge works out the task from the customer's message. For "refund my sale item
from 20 days ago" it decides the task is *a refund*, and marks the agent down for refusing,
although refusing is the right outcome. `TaskCompletionMetric` takes a `task` argument for this:
here it gets the case's `reference_answer`, the outcome the policy requires.

In [ ]:
# ============================================================================
# TaskCompletionMetric again, with the correct outcome given as the task
# ============================================================================
# 💰 Another 60 cases × 2 judge calls with gpt-4o: about $0.40.
for case in state_cases:
    run = runs_by_case[case["id"]]
    test_case = LLMTestCase(
        input=case["input"],
        actual_output=run["answer"],
        tools_called=[ToolCall(name=call["name"], input_parameters=call["args"], output=result)
                      for call, result in zip(run["tool_calls"], replayed_results[case["id"]])],
    )
    task_completion_with_task = TaskCompletionMetric(task=case["reference_answer"], model=JUDGE_MODEL, async_mode=False)
    task_completion_with_task.measure(test_case, _show_indicator=False)
    scores[case["id"]]["judge_task_completion_given_task"] = task_completion_with_task.score

judge_results = [(case["slice"], scores[case["id"]]["judge_task_completion_given_task"] >= 0.5) for case in state_cases]
print("⚖️ TaskCompletion judge, given the correct outcome as the task")
print_pass_rate_by_slice(judge_results)

---
## ⚖️ Part 4: RAGAS `AgentGoalAccuracyWithReference`

RAGAS asks a slightly different question: given the conversation (as messages, including the
tool calls and their results) and a **reference** describing the right outcome, was the goal
reached? It answers 1 or 0. The reference is the case's `reference_answer`. RAGAS metrics are
asynchronous, so the call uses `await`, which works directly in a notebook cell. It runs on a
20-case subset: every case the state check failed, then passing cases in order.

In [ ]:
# ============================================================================
# RAGAS: turn a run into messages (customer → tool calls → tool results → reply)
# ============================================================================
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.metrics.collections import AgentGoalAccuracyWithReference
from ragas.messages import HumanMessage as RagasHumanMessage, AIMessage as RagasAIMessage
from ragas.messages import ToolMessage as RagasToolMessage, ToolCall as RagasToolCall

def run_as_ragas_messages(case):
    run = runs_by_case[case["id"]]
    messages = [RagasHumanMessage(content=case["input"])]
    for call, result in zip(run["tool_calls"], replayed_results[case["id"]]):
        messages.append(RagasAIMessage(content="", tool_calls=[RagasToolCall(name=call["name"], args=call["args"])]))
        messages.append(RagasToolMessage(content=json.dumps(result)))
    messages.append(RagasAIMessage(content=run["answer"]))
    return messages

goal_accuracy = AgentGoalAccuracyWithReference(llm=llm_factory(JUDGE_MODEL, client=AsyncOpenAI()))
failed_first = [case for case in state_cases if not scores[case["id"]]["end_state_correct"]]
passed_next = [case for case in state_cases if scores[case["id"]]["end_state_correct"]]
ragas_subset = (failed_first + passed_next)[:20]
print(f"✅ RAGAS subset: {len(ragas_subset)} cases ({len(failed_first)} that failed the state check)")

In [ ]:
# ============================================================================
# RAGAS AgentGoalAccuracyWithReference on the subset
# ============================================================================
# 💰 20 judge calls with gpt-4o: about $0.10.
for case in ragas_subset:
    result = await goal_accuracy.ascore(user_input=run_as_ragas_messages(case), reference=case["reference_answer"])
    scores[case["id"]]["judge_goal_accuracy"] = result.value

ragas_results = [(case["slice"], scores[case["id"]]["judge_goal_accuracy"] >= 0.5) for case in ragas_subset]
print("⚖️ RAGAS says the goal was reached")
print_pass_rate_by_slice(ragas_results)

In [ ]:
# ============================================================================
# BOTH JUDGES AGAINST THE GROUND TRUTH
# ============================================================================
def agreement_with_state(score_name, cases):
    agree, disagreements = 0, []
    for case in cases:
        judge_pass = scores[case["id"]][score_name] >= 0.5
        if judge_pass == scores[case["id"]]["end_state_correct"]:
            agree += 1
        else:
            disagreements.append((case["id"], judge_pass))
    return agree, disagreements

for score_name, cases in [("judge_task_completion", state_cases), ("judge_task_completion_given_task", state_cases),
                          ("judge_goal_accuracy", ragas_subset)]:
    agree, disagreements = agreement_with_state(score_name, cases)
    print(f"⚖️ {score_name}: agrees with the end-state check on {agree} of {len(cases)} ({agree / len(cases):.0%})")
    for case_id, judge_pass in disagreements:
        state = "passed" if scores[case_id]["end_state_correct"] else "FAILED"
        print(f"   {case_id:<20} store {state:<7} judge says {'done' if judge_pass else 'not done'}")
print("Reading: each disagreement is a case where trusting the judge would mislead you. Telling the")
print("judge what 'done' means closes much of the gap; measuring it against a state check, where one")
print("exists, is how you find out how much.")

---
## 💬 Part 5: From messages to conversations

The app answers one message at a time. Real customers send several: "I want to return my tent",
then the order ID, then the email. Evaluating that needs a conversation, and a way to give the
app the history without changing it.

The app's state has two inputs: `question`, which the supervisor routes and the RAG agent answers,
and `messages`, which the account agent reads. The small wrapper below fills both: `question`
gets the conversation so far plus the new message, so routing sees the context, and `messages`
gets every earlier turn, so the account agent remembers what it was told. The store is reset at
the start of each conversation, not each turn, so a refund in turn 3 is still there in turn 4.

In [ ]:
# ============================================================================
# CHAT WRAPPER: one customer turn, with the conversation so far
# ============================================================================
from langchain_core.messages import AIMessage

def chat_turn(history, user_message):
    """history: earlier turns as [{"role": "user" or "assistant", "content": "..."}].
    Returns the agent's reply and the route the supervisor chose."""
    earlier_text = "".join(f"{turn['role']}: {turn['content']}\n" for turn in history)
    question = user_message if not history else \
        f"Conversation so far:\n{earlier_text}\nThe customer now says: {user_message}"
    messages = []
    for turn in history:
        messages.append(HumanMessage(content=turn["content"]) if turn["role"] == "user" else AIMessage(content=turn["content"]))
    messages.append(HumanMessage(content=user_message))
    final_state = support_agent.invoke({"question": question, "messages": messages},
                                       config={"run_name": "support-chat", "tags": ["notebook-08"]})
    return final_state["answer"], final_state["route"]

print("✅ chat_turn ready")

### 5.1 📜 Scripted conversations

Four conversations, each written to test one thing that only shows up across turns. Each also
says what a good outcome is, which DeepEval's completeness metric and the end-state check use.

In [ ]:
# ============================================================================
# FOUR SCRIPTED CONVERSATIONS
# ============================================================================
scripted_conversations = [
    {"name": "refund-across-turns",
     "user_turns": ["I'd like to return the tent I bought, it's too small.", "The order is PC-10001.", "My email is maya@example.com."],
     "expected_outcome": "The tent in PC-10001 is refunded, without asking again for the order ID.",
     "expected_refunds": ["PC-10001"]},
    {"name": "policy-then-action",
     "user_turns": ["How many days do I have to return an item?", "Then please refund order PC-10006, my email is maya@example.com."],
     "expected_outcome": "Explains the 30-day window, then hands PC-10006 (outside it) to a human instead of refunding.",
     "expected_refunds": []},
    {"name": "topic-switch",
     "user_turns": ["How much is express shipping?", "Can I change the delivery address on order PC-10003? My email is lee@example.com."],
     "expected_outcome": "Gives the express price, then says a shipped order's address can't be changed and to contact the carrier.",
     "expected_refunds": []},
    {"name": "stays-in-role",
     "user_turns": ["Can you help me write a cover letter for a job?", "OK. What's the warranty on a backpack, then?"],
     "expected_outcome": "Declines the cover letter, then gives the 1-year backpack warranty.",
     "expected_refunds": []},
]
print(f"📜 {len(scripted_conversations)} scripted conversations")

In [ ]:
# ============================================================================
# RUN THE SCRIPTED CONVERSATIONS
# ============================================================================
# 💰 9 turns with gpt-4o-mini: well under a cent. Each turn is a trace in LangSmith.
for conversation in scripted_conversations:
    database = reset_database()                 # one fresh store per conversation
    history, routes = [], []
    print(f"💬 {conversation['name']}")
    for user_message in conversation["user_turns"]:
        reply, route = chat_turn(history, user_message)
        history.append({"role": "user", "content": user_message})
        history.append({"role": "assistant", "content": reply})
        routes.append(route)
        print(f"   👤 {user_message}\n   🤖 [{route}] {reply}")
    conversation["turns"] = history
    conversation["routes"] = routes
    conversation["refunds_issued"] = [refund["order_id"] for refund in database["refunds"]]
    conversation["tickets_created"] = len(database["tickets"])
    print(f"   🏁 refunds {conversation['refunds_issued']}   tickets {conversation['tickets_created']}\n")

### 5.2 📏 Four conversational metrics

| Metric | Question |
|---|---|
| `KnowledgeRetentionMetric` | Did the agent remember what the customer already told it? |
| `ConversationCompletenessMetric` | Was everything the customer asked for dealt with by the end? |
| `TurnRelevancyMetric` | Was each reply relevant, given the turns before it? |
| `RoleAdherenceMetric` | Did the agent stay in its role (`chatbot_role`) throughout? |

Each takes a `ConversationalTestCase`: the turns, plus the role the bot is meant to play.

In [ ]:
# ============================================================================
# CONVERSATIONAL METRICS on the scripted conversations
# ============================================================================
# 💰 4 conversations × 4 metrics with gpt-4o: about $0.30.
from deepeval.test_case import ConversationalTestCase, Turn
from deepeval.metrics import KnowledgeRetentionMetric, ConversationCompletenessMetric
from deepeval.metrics import TurnRelevancyMetric, RoleAdherenceMetric

CHATBOT_ROLE = ("Support assistant for Pinecrest Outdoor Supply: answers from the support articles, "
                "handles orders and refunds after verifying the order ID and email, and hands hard cases to a human.")
conversation_metrics = {
    "knowledge_retention": KnowledgeRetentionMetric(model=JUDGE_MODEL, async_mode=False),
    "completeness": ConversationCompletenessMetric(model=JUDGE_MODEL, async_mode=False),
    "turn_relevancy": TurnRelevancyMetric(model=JUDGE_MODEL, async_mode=False),
    "role_adherence": RoleAdherenceMetric(model=JUDGE_MODEL, async_mode=False),
}

def score_conversation(turns, expected_outcome):
    test_case = ConversationalTestCase(turns=[Turn(role=turn["role"], content=turn["content"]) for turn in turns],
                                       chatbot_role=CHATBOT_ROLE, expected_outcome=expected_outcome)
    conversation_scores = {}
    for metric_name, metric in conversation_metrics.items():
        metric.measure(test_case, _show_indicator=False)
        conversation_scores[metric_name] = round(metric.score, 2)
        if metric.score < 0.5:
            print(f"      ⚠️ {metric_name} {metric.score:.2f}: {metric.reason[:160]}")   # why it scored low
    return conversation_scores

for conversation in scripted_conversations:
    print(f"📏 {conversation['name']}")
    conversation["scores"] = score_conversation(conversation["turns"], conversation["expected_outcome"])
    state_ok = conversation["refunds_issued"] == conversation["expected_refunds"]
    print(f"   {conversation['scores']}   end state {'✅' if state_ok else '❌'}")
print("Reading: each metric looks at one property of the whole conversation; read the reasons for")
print("low scores, and the end state next to them: a polite conversation can still refund the wrong order.")

---
## 🤖 Part 6: A simulated customer

Scripted conversations test what you thought of. DeepEval's `ConversationSimulator` plays the
customer with a model (`gpt-4o-mini` here): it is given a **scenario**, a description of the
customer and the **expected outcome**, and writes each next message from the agent's last reply,
until the outcome is reached or the turn limit is hit. Each simulated conversation is new, so the
same goldens give a fresh test every run.

The simulator calls a function with the new message and the turns so far; ours passes them to
`chat_turn`. The store is reset whenever a new conversation starts.

In [ ]:
# ============================================================================
# THE SIMULATOR: goldens, and the callback that connects it to the agent
# ============================================================================
from deepeval.dataset import ConversationalGolden
from deepeval.simulator import ConversationSimulator

simulation_goldens = [
    ConversationalGolden(
        scenario="A customer wants a refund for a tent that is too small, but at first forgets to give the order ID.",
        user_description="Maya, order PC-10001, email maya@example.com. Polite, gives one detail at a time.",
        expected_outcome="The tent is refunded after the agent gets both the order ID and the email."),
    ConversationalGolden(
        scenario="A customer asks for a refund on a $649 sleeping bag and pushes back when told it needs a human.",
        user_description="Ana, order PC-10004, email ana@example.com. Impatient, wants it done now.",
        expected_outcome="No refund is issued; a ticket goes to a human agent, and the customer is told when to expect a reply."),
]

def model_callback(input, turns):
    """Called by the simulator with the customer's new message and every turn so far."""
    global database
    history = [{"role": turn.role, "content": turn.content} for turn in turns]
    if history and history[-1]["role"] == "user" and history[-1]["content"] == input:
        history = history[:-1]                  # the new message is passed separately
    if not history:
        database = reset_database()             # a new conversation starts with a fresh store
    reply, route = chat_turn(history, input)
    return Turn(role="assistant", content=reply)

# One conversation at a time: they share the one mock store, so running two at once would mix them up.
simulator = ConversationSimulator(model_callback=model_callback, simulator_model=SIMULATED_USER_MODEL, max_concurrent=1)
print(f"✅ Simulator ready: {len(simulation_goldens)} goldens")

In [ ]:
# ============================================================================
# SIMULATE, THEN SCORE WITH THE SAME FOUR METRICS
# ============================================================================
# 💰 Up to 2 × 5 turns of gpt-4o-mini (customer and agent), then 2 × 4 metrics with gpt-4o: about $0.20.
simulated_cases = simulator.simulate(conversational_goldens=simulation_goldens, max_user_simulations=5)

simulated_conversations = []
for golden, simulated in zip(simulation_goldens, simulated_cases):
    turns = [{"role": turn.role, "content": turn.content} for turn in simulated.turns]
    print(f"🤖 {golden.scenario}")
    for turn in turns:
        print(f"   {'👤' if turn['role'] == 'user' else '🤖'} {turn['content']}")
    conversation_scores = score_conversation(turns, golden.expected_outcome)
    print(f"   📏 {conversation_scores}\n")
    simulated_conversations.append({"scenario": golden.scenario, "turns": turns, "scores": conversation_scores})
print("Reading: a simulated customer finds paths nobody scripted; the price is that each run is")
print("different, so compare averages over several runs, not one conversation against another.")

---
## 💾 Part 7: Save the scores

Per-case scores go into `data/agent_scores.json`, beside those from notebooks 06 and 07.
Conversations are not golden-set cases, so their scores go into `data/conversation_scores.json`.

In [ ]:
# ============================================================================
# SAVE: per-case scores, and the conversations with their scores
# ============================================================================
SCORES_PATH = os.path.join("data", "agent_scores.json")
agent_scores = {}
if os.path.exists(SCORES_PATH):
    with open(SCORES_PATH, encoding="utf-8") as f:
        agent_scores = json.load(f)
for case_id, case_scores in scores.items():
    run = runs_by_case[case_id]
    record = agent_scores.get(case_id, {"case_id": case_id, "run_id": run["run_id"], "slice": run["slice"]})
    record.update(case_scores)
    agent_scores[case_id] = record
with open(SCORES_PATH, "w", encoding="utf-8") as f:
    json.dump(agent_scores, f, indent=2)

conversation_records = {"scripted": scripted_conversations, "simulated": simulated_conversations}
with open(os.path.join("data", "conversation_scores.json"), "w", encoding="utf-8") as f:
    json.dump(conversation_records, f, indent=2, ensure_ascii=False)
print(f"✅ Saved task-completion scores for {len(scores)} cases → {SCORES_PATH}")
print(f"✅ Saved {len(scripted_conversations)} scripted and {len(simulated_conversations)} simulated conversations → data/conversation_scores.json")

---
## 📝 Summary

In this notebook, we learned:

### 1. Task completion
- **The end-state check is ground truth**: when a task leaves a mark in the store, check the mark
- **Judges are measured against it**: `TaskCompletionMetric` and RAGAS goal accuracy, with every disagreement listed

### 2. Conversations
- **A wrapper, not a rewrite**: history goes into the app's existing `question` and `messages` inputs
- **Scripted conversations** test the multi-turn behaviour you thought of; the **simulator** finds the rest
- **Four metrics**: retention, completeness, turn relevancy and role adherence, read next to the end state

### Next Steps
- Notebook 09 attacks the agent on purpose: prompt injection, data leaks and refunds it must never issue